# 🚀 Bit-MC-SSM: Kaggle 2x T4 Multi-GPU (torchrun DDP) Scale-Up Training
### 〜 事前トークナイズ（Tiktoken） ＋ 2x T4 GPU ＋ `torchrun` (DDP) ＋ ゼロコピー Memmap 学習（40,000+ tok/s）〜

本ノートブックは、**Kaggle の 2x T4 GPU（週30時間無料・1セッション12時間連続）** をフル稼働させ、
1. **事前トークナイズ（`preprocess_data.py`）**: Tiktoken により 50,000 件のテキストをわずか 30 秒で `train_tokens.bin` に変換（CPU 待機ゼロ化）
2. **分散並列学習（`torchrun DDP`）**: PyTorch 公式の標準分散ランチャー ＋ GaLore ＋ 1.58-bit BitNet v2 ＋ Triton Fused カーネルにより **40,000+ tokens/sec** で学習
3. **チェックポイント保存**: 毎エポックの `ckpt_last.pt` / `ckpt_best.pt` を自動保存
4. **2-bit バイナリ出力**: CPU で 100+ tokens/s で動く `model_medium-30M.bin` を即座にエクスポート

> 💡 **Kaggle 設定手順:**
> 画面右上の **Settings -> Accelerator -> GPU T4 x2** を選択してください。

## 1. 2x T4 GPU の認識 ＆ 依存ライブラリのインストール

In [ ]:
!pip install -q transformers datasets einops tiktoken tqdm triton
# Tri Dao causal-conv1d prebuilt wheel for Python 3.12 + CUDA 12
!pip install -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl || pip install -q causal-conv1d --no-build-isolation || true

import torch
num_gpus = torch.cuda.device_count()
print(f"🚀 PyTorch Version: {torch.__version__}")
print(f"🎮 Detected GPUs: {num_gpus} device(s)")

for i in range(num_gpus):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"   GPU [{i}]: {name} ({vram:.2f} GB VRAM)")

if num_gpus < 2:
    print("⚠️ Notice: Set Accelerator to 'GPU T4 x2' in Kaggle Settings panel for 2x speed.")

## 2. リポジトリのクローン（または最新コードの取得）

In [ ]:
# GitHub リポジトリから最新コードを取得
![ -d 'BitMC-SSM' ] || git clone https://github.com/fukayatti/BitMC-SSM.git
%cd BitMC-SSM
!git pull origin main

## 3. ⚡ 高速事前トークナイズ（Pre-tokenization to `.bin`）
HuggingFace からストリーミング取得し、Tiktoken で一括トークナイズして `uint16` 配列（`.bin`）に保存します。
これにより、**GPU 学習中の CPU トークナイズ待ちがゼロ（GPU 使用率 100% 維持）** になります。

In [ ]:
# TinyStories または SmolLM を事前トークナイズ（50,000 件 / 約 30 秒で完了）
!python python/preprocess_data.py \
    --dataset tinystories \
    --num_samples 50000 \
    --out /kaggle/working/train_tokens.bin

## 4. 🔥 `torchrun` による 2x T4 分散並列学習の実行（ゼロコピー Memmap）
`--data_bin /kaggle/working/train_tokens.bin` を指定することで、起動待ち時間 0.01 秒で学習が開始されます。
万が一途中で中断した場合も、`--resume_from /kaggle/working/checkpoints/ckpt_last.pt` を指定するだけで続きから再開できます。

In [ ]:
# プリセット設定（30M 推奨）: 
# - medium-30M:  --d_model 384 --n_layers 8  --d_state 32 --batch_size 32 --grad_accum_steps 1
# - large-60M:   --d_model 512 --n_layers 8  --d_state 64 --batch_size 24 --grad_accum_steps 2
# - base-135M:   --d_model 768 --n_layers 12 --d_state 64 --batch_size 16 --grad_accum_steps 2

!torchrun --nproc_per_node=2 python/train.py \
    --data_bin /kaggle/working/train_tokens.bin \
    --seq_len 128 \
    --d_model 384 \
    --n_layers 8 \
    --d_state 32 \
    --batch_size 48 \
    --grad_accum_steps 1 \
    --chunk_size 64 \
    --num_workers 2 \
    --lr 1.0e-3 \
    --galore_rank 16 \
    --epochs 8 \
    --amp \
    --compile \
    --save_ckpt_dir /kaggle/working/checkpoints \
    --save_every_epochs 1 \
    --out_bin /kaggle/working/model_medium-30M.bin

# Vocab の出力
from transformers import GPT2TokenizerFast
import json
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
with open('/kaggle/working/vocab.json', 'w', encoding='utf-8') as f:
    json.dump(tokenizer.get_vocab(), f, ensure_ascii=False)

print("🎉 2x T4 torchrun training & 2-bit binary export finished successfully!")

## 5. 2-bit バイナリの検証 ＆ C++ エンジンでのローカル推論テスト
Kaggle 環境内で C++ 推論エンジン（`./infer`）をコンパイルし、出力された `model_medium-30M.bin` で **ゼロ乗算（Zero-GEMM）推論** をテストします。

In [ ]:
# C++ 推論エンジンのビルド
!make clean && make -j4

# 2-bit バイナリ推論テスト（サンプリング指定: temp=0.7, top_p=0.9, rep_penalty=1.15）
!./infer /kaggle/working/model_medium-30M.bin 60 "Once upon a time, Lily found a" 0.7 40 0.9 1.15

## 6. Kaggle Output パネルからのダウンロード
学習が完了すると、右側サイドバーの **「Output」パネル** に以下のファイルが出力されます：
- `/kaggle/working/model_medium-30M.bin`（CPU L3 キャッシュに収まる超軽量 2-bit バイナリ）
- `/kaggle/working/vocab.json`
- `/kaggle/working/checkpoints/ckpt_best.pt`（最高精度の PyTorch 重み）
- `/kaggle/working/checkpoints/ckpt_last.pt`（最終エポックのチェックポイント）

右側の `...` ボタンから **「Download」** をクリックして手元の PC に保存し、手元の CPU（`./infer`）でいつでも爆速推論が可能です！